# BEiT3 keyframe embeddings on Kaggle (T4)

This notebook builds the active retrieval artifacts for this backend: normalized 1024-dimensional BEiT3 image embeddings, an ID-mapped FAISS index, and metadata aligned to the index. It does **not** build the legacy CLIP index.

Before running, attach these Kaggle Input datasets:

1. Keyframes with layout `keyframes_root/<split>/<video_id>/<frame>.webp`.
2. The BEiT3 Large Retrieval checkpoint (`beit3_large_patch16_384_retrieval`).
3. Optional `map-keyframes` CSV dataset. It preserves real source timestamps instead of using `frame_number / 25`.

The output folder is self-contained except for source images. Download it after completion, keep the same keyframe tree on the backend machine, and point `BEIT3_*` environment variables at the generated artifacts.

In [ ]:
# Kaggle T4 runtime dependencies. Run once after enabling Accelerator: GPU T4 x2 or GPU T4 x1.
# The output FAISS index remains CPU-portable for the Windows backend.
%pip install -q --disable-pip-version-check torchscale sentencepiece pyarrow faiss-cpu

import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import json
import re
import csv
import time
import shutil
from collections import defaultdict
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from torchscale.architecture.config import EncoderConfig
from torchscale.model.BEiT3 import BEiT3

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator, then restart the session.'
DEVICE = torch.device('cuda:0')
torch.backends.cudnn.benchmark = True
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

In [ ]:
# Configuration: change only these paths for your Kaggle inputs.
# Do not use /kaggle/working for KEYFRAMES_ROOT: Kaggle inputs are read-only and safer for long runs.
KEYFRAMES_ROOT = Path('/kaggle/input/aic-keyframes/keyframes_L21_onwards')
CHECKPOINT_PATH = Path('/kaggle/input/aic-beit3-model/beit3_large_patch16_384_retrieval.pth')
MAP_KEYFRAMES_DIR = Path('/kaggle/input/aic-map-keyframes/map-keyframes')  # Set to None when unavailable.
OUTPUT_ROOT = Path('/kaggle/working/beit3_keyframe_artifacts')

# T4-safe default. Increase carefully only after a one-video test succeeds.
BATCH_SIZE = 32
DEFAULT_FPS = 25.0
INCLUDE_SPLITS = None          # Example: {'L21_a', 'L21_b'} for a distributed Kaggle job.
TEST_ONLY_VIDEOS = 0           # Set to 1 for a smoke test; set back to 0 for all videos.
RESUME = True                  # Reuses per-video .npy files only when their manifest matches.
WRITE_METADATA_BEIT3_JSON = True

assert KEYFRAMES_ROOT.is_dir(), f'Missing keyframes root: {KEYFRAMES_ROOT}'
assert CHECKPOINT_PATH.is_file(), f'Missing checkpoint: {CHECKPOINT_PATH}'
if MAP_KEYFRAMES_DIR is not None and not MAP_KEYFRAMES_DIR.is_dir():
    print(f'WARNING: map-keyframes is unavailable: {MAP_KEYFRAMES_DIR}. Timestamps will fall back to frame/fps.')
    MAP_KEYFRAMES_DIR = None

EMBEDDINGS_DIR = OUTPUT_ROOT / 'embeddings'
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
print('Keyframes:', KEYFRAMES_ROOT)
print('Checkpoint:', CHECKPOINT_PATH)
print('Output:', OUTPUT_ROOT)

In [ ]:
# Exact inference-only architecture used by src/utils/beit3_backbone.py in this backend.
# Keep this configuration unchanged: the retrieval checkpoint and backend text encoder require it.
def build_large_retrieval_config(img_size=384, vocab_size=64010):
    return EncoderConfig(
        img_size=img_size, patch_size=16, vocab_size=vocab_size, multiway=True,
        layernorm_embedding=False, normalize_output=True, no_output_layer=True,
        drop_path_rate=0, encoder_embed_dim=1024, encoder_attention_heads=16,
        encoder_ffn_embed_dim=4096, encoder_layers=24, checkpoint_activations=False,
    )

class BEiT3ForRetrieval(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.args = config
        self.beit3 = BEiT3(config)
        self.language_head = nn.Linear(1024, 1024, bias=False)
        self.vision_head = nn.Linear(1024, 1024, bias=False)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, image=None, text_description=None, padding_mask=None, only_infer=True):
        if not only_infer:
            raise NotImplementedError('Inference-only notebook.')
        vision_cls = language_cls = None
        if image is not None:
            output = self.beit3(textual_tokens=None, visual_tokens=image, text_padding_position=None)
            vision_cls = F.normalize(self.vision_head(output['encoder_out'][:, 0, :]), dim=-1)
        if text_description is not None:
            output = self.beit3(textual_tokens=text_description, visual_tokens=None, text_padding_position=padding_mask)
            language_cls = F.normalize(self.language_head(output['encoder_out'][:, 0, :]), dim=-1)
        return vision_cls, language_cls

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = checkpoint.get('model', checkpoint) if isinstance(checkpoint, dict) else checkpoint
model = BEiT3ForRetrieval(build_large_retrieval_config()).eval().to(DEVICE)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
assert not missing and not unexpected, (f'Checkpoint architecture mismatch. Missing={missing[:5]}, unexpected={unexpected[:5]}')
print('Loaded BEiT3 Large Retrieval checkpoint; output dimension = 1024.')

In [ ]:
IMAGE_EXTENSIONS = {'.webp', '.jpg', '.jpeg', '.png'}
MEAN = torch.tensor((0.5, 0.5, 0.5)).view(3, 1, 1)
STD = torch.tensor((0.5, 0.5, 0.5)).view(3, 1, 1)

def natural_key(path):
    return [int(chunk) if chunk.isdigit() else chunk.lower() for chunk in re.split(r'(\d+)', path.name)]

def parse_frame_idx(path):
    numbers = re.findall(r'\d+', path.stem)
    return int(numbers[-1]) if numbers else None

def load_map_keyframes(map_root):
    maps = {}
    if map_root is None:
        return maps
    for csv_path in sorted(map_root.glob('*.csv')):
        by_idx, by_n, fps = {}, {}, None
        with csv_path.open('r', encoding='utf-8-sig', newline='') as handle:
            for row in csv.DictReader(handle):
                try:
                    item = {'frame_idx': int(row['frame_idx']), 'timestamp': float(row['pts_time']), 'fps': float(row.get('fps') or DEFAULT_FPS), 'n': int(row['n'])}
                except (KeyError, TypeError, ValueError):
                    continue
                by_idx[item['frame_idx']] = item
                by_n[item['n']] = item
                fps = item['fps']
        if by_idx:
            maps[csv_path.stem] = {'by_idx': by_idx, 'by_n': by_n, 'fps': fps or DEFAULT_FPS}
    print(f'Loaded timestamp maps for {len(maps)} videos.')
    return maps

def timestamp_info(video_id, image_path, ordinal, maps):
    frame_idx = parse_frame_idx(image_path)
    mapping = maps.get(video_id)
    if mapping and frame_idx in mapping['by_idx']:
        item = mapping['by_idx'][frame_idx]
        return item['frame_idx'], item['timestamp'], item['fps'], 'map_frame_idx_exact'
    if mapping and ordinal + 1 in mapping['by_n']:
        item = mapping['by_n'][ordinal + 1]
        return item['frame_idx'], item['timestamp'], item['fps'], 'map_keyframe_number_exact'
    fps = (mapping or {}).get('fps', DEFAULT_FPS)
    frame_idx = ordinal if frame_idx is None else frame_idx
    return frame_idx, frame_idx / fps, fps, 'filename_frame_number_fallback'

def discover_videos(root):
    videos = []
    for split_dir in sorted((p for p in root.iterdir() if p.is_dir()), key=lambda p: p.name):
        if INCLUDE_SPLITS is not None and split_dir.name not in INCLUDE_SPLITS:
            continue
        for video_dir in sorted((p for p in split_dir.iterdir() if p.is_dir()), key=lambda p: p.name):
            frames = sorted((p for p in video_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS), key=natural_key)
            if frames:
                videos.append((split_dir.name, video_dir.name, frames))
    return videos

maps = load_map_keyframes(MAP_KEYFRAMES_DIR)
videos = discover_videos(KEYFRAMES_ROOT)
if TEST_ONLY_VIDEOS:
    videos = videos[:TEST_ONLY_VIDEOS]
print(f'Discovered {len(videos)} videos and {sum(len(frames) for _, _, frames in videos):,} keyframes.')
assert videos, 'No keyframes found. Check KEYFRAMES_ROOT and INCLUDE_SPLITS.'

In [ ]:
def image_tensor(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert('RGB').resize((384, 384), Image.Resampling.BICUBIC)
        array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)
    return (tensor - MEAN) / STD

def encode_video(frame_paths):
    vectors = []
    for start in range(0, len(frame_paths), BATCH_SIZE):
        batch_paths = frame_paths[start:start + BATCH_SIZE]
        batch = torch.stack([image_tensor(path) for path in batch_paths]).to(DEVICE, non_blocking=True)
        with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.float16):
            image_features, _ = model(image=batch, only_infer=True)
        vectors.append(image_features.float().cpu().numpy())
        del batch, image_features
    features = np.concatenate(vectors, axis=0).astype(np.float32, copy=False)
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    features /= np.clip(norms, 1e-12, None)
    assert features.shape == (len(frame_paths), 1024)
    return features

manifest_path = OUTPUT_ROOT / 'embedding_manifest.json'
run_manifest = {
    'model_checkpoint_name': CHECKPOINT_PATH.name,
    'embedding_dim': 1024,
    'image_size': 384,
    'normalization': 'ImageNet inception mean/std = 0.5/0.5',
    'keyframes_root_name': KEYFRAMES_ROOT.name,
}
previous_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
if RESUME and previous_manifest is not None:
    assert previous_manifest == run_manifest, 'Existing output was made with another model/config. Use a new OUTPUT_ROOT.'

records, video_rows, failures = [], [], []
vector_id = 0
started = time.perf_counter()
for split, video_id, frame_paths in tqdm(videos, desc='Embedding videos'):
    output_name = f'{split}_{video_id}_embeddings.npy'
    output_path = EMBEDDINGS_DIR / output_name
    features = None
    if RESUME and output_path.exists():
        candidate = np.load(output_path, mmap_mode='r')
        if candidate.shape == (len(frame_paths), 1024):
            features = np.asarray(candidate, dtype=np.float32)
    if features is None:
        try:
            features = encode_video(frame_paths)
            np.save(output_path, features)
        except Exception as exc:
            if 'out of memory' in str(exc).lower():
                torch.cuda.empty_cache()
                raise RuntimeError(f'CUDA OOM on {video_id}. Reduce BATCH_SIZE from {BATCH_SIZE} and resume.') from exc
            failures.append({'split': split, 'video_id': video_id, 'error': repr(exc)})
            print(f'WARNING: skipping {video_id}: {exc}')
            continue

    start_vector_id = vector_id
    for ordinal, image_path in enumerate(frame_paths):
        source_idx, timestamp, fps, timestamp_source = timestamp_info(video_id, image_path, ordinal, maps)
        records.append({
            'vector_id': vector_id, 'video_id': video_id, 'frame_id': f'{source_idx:06d}',
            'frame_path': image_path.relative_to(KEYFRAMES_ROOT).as_posix(),
            'parent_namespace': split, 'timestamp': float(timestamp), 'timestamp_s': float(timestamp),
            'fps': float(fps), 'source_frame_idx': int(source_idx),
            'keyframe_number': ordinal + 1, 'timestamp_source': timestamp_source,
        })
        vector_id += 1
    video_rows.append({
        'video_id': video_id, 'parent_namespace': split, 'video_namespace': video_id,
        'artifact_base': f'{split}_{video_id}', 'embedding_file': output_name,
        'frame_count': len(frame_paths), 'embedding_dim': 1024, 'first_vector_id': start_vector_id,
    })

elapsed = time.perf_counter() - started
assert records, 'No embeddings were written.'
print(f'Encoded/reused {len(records):,} frames from {len(video_rows)} videos in {elapsed / 60:.1f} minutes.')
if failures:
    pd.DataFrame(failures).to_csv(OUTPUT_ROOT / 'failed_videos.csv', index=False)
    print(f'WARNING: {len(failures)} videos were skipped. See failed_videos.csv and fix them before deployment.')

In [ ]:
# Build a portable IndexIDMap2(IndexFlatIP). IDs exactly match global_ids.parquet vector_id.
global_ids = pd.DataFrame.from_records(records)
video_metadata = pd.DataFrame.from_records(video_rows)
assert global_ids['vector_id'].tolist() == list(range(len(global_ids)))

index = faiss.IndexIDMap2(faiss.IndexFlatIP(1024))
for video in tqdm(video_rows, desc='Building FAISS index'):
    features = np.load(EMBEDDINGS_DIR / video['embedding_file']).astype(np.float32, copy=False)
    ids = np.arange(video['first_vector_id'], video['first_vector_id'] + video['frame_count'], dtype=np.int64)
    assert len(features) == len(ids)
    index.add_with_ids(features, ids)

assert index.ntotal == len(global_ids), (index.ntotal, len(global_ids))
global_ids.to_parquet(OUTPUT_ROOT / 'global_ids.parquet', index=False)
video_metadata.to_parquet(OUTPUT_ROOT / 'video_metadata.parquet', index=False)
faiss.write_index(index, str(OUTPUT_ROOT / 'beit3_faiss.index'))

index_meta = {
    'model': 'beit3_large_patch16_384_retrieval', 'embedding_dim': 1024,
    'metric': 'inner_product_on_l2_normalized_vectors', 'vector_count': int(index.ntotal),
    'video_count': int(len(video_metadata)), 'keyframes_root_layout': '<split>/<video_id>/<frame_name>',
    'created_by': 'scripts/notebooks/beit3_kaggle_keyframe_embeddings.ipynb',
}
(OUTPUT_ROOT / 'index_meta.json').write_text(json.dumps(index_meta, indent=2) + '\n', encoding='utf-8')
manifest_path.write_text(json.dumps(run_manifest, indent=2) + '\n', encoding='utf-8')

if WRITE_METADATA_BEIT3_JSON:
    metadata_beit3 = {}
    for row in global_ids.to_dict('records'):
        item = dict(row)
        item['faiss_id'] = int(item['vector_id'])
        item['split'] = item['parent_namespace']
        item['frame_name'] = Path(item['frame_path']).name
        item['frame_index'] = int(item['keyframe_number']) - 1
        item['global_frame_id'] = int(item['source_frame_idx'])
        metadata_beit3[str(item['vector_id'])] = item
    (OUTPUT_ROOT / 'metadata_beit3.json').write_text(json.dumps(metadata_beit3, ensure_ascii=False) + '\n', encoding='utf-8')

print('FAISS vectors:', index.ntotal)
print('Artifacts:', sorted(path.name for path in OUTPUT_ROOT.iterdir()))

In [ ]:
# Final integrity check. Do not copy artifacts to the backend unless this cell passes.
loaded_index = faiss.read_index(str(OUTPUT_ROOT / 'beit3_faiss.index'))
loaded_ids = pd.read_parquet(OUTPUT_ROOT / 'global_ids.parquet')
assert loaded_index.d == 1024
assert loaded_index.ntotal == len(loaded_ids) == len(records)
assert loaded_ids['vector_id'].is_unique
assert loaded_ids['vector_id'].tolist() == list(range(len(loaded_ids)))
assert loaded_ids['frame_path'].str.contains('/').all()

# Spot-check that FAISS returns the self vector as rank 1.
sample_id = min(17, len(loaded_ids) - 1)
sample = loaded_ids.iloc[sample_id]
video = next(row for row in video_rows if row['video_id'] == sample.video_id)
features = np.load(EMBEDDINGS_DIR / video['embedding_file'])
local_index = int(sample.keyframe_number) - 1
score, found = loaded_index.search(features[local_index:local_index + 1].astype(np.float32), 1)
assert int(found[0, 0]) == int(sample.vector_id), (sample.vector_id, found[0, 0])
print('PASS: dimension=1024, vectors=', loaded_index.ntotal, 'self-similarity=', float(score[0, 0]))

print('\nBackend .env values after downloading artifacts:')
print('BEIT3_FAISS_INDEX_PATH=<artifact_dir>/beit3_faiss.index')
print('BEIT3_GLOBAL_IDS_PATH=<artifact_dir>/global_ids.parquet')
print('BEIT3_VIDEO_METADATA_PATH=<artifact_dir>/video_metadata.parquet')
print('BEIT3_INDEX_META_PATH=<artifact_dir>/index_meta.json')
print('BEIT3_CHECKPOINT_PATH=<same BEiT3 checkpoint used above>')
print('BEIT3_TOKENIZER_PATH=<the matching SentencePiece tokenizer.model>')
print('BEIT3_DEVICE=cuda')